Zadanie zaczynamy od sparsowania plików z danymi oraz załadowania ich do  Dataframeów z biblioteki Pandas. 
Usuwamy pliki **agency.txt** oraz **feed_info.txt**, jako że nie zawierają potrzebnych informacji

In [56]:
import os
import pandas as pd 
from pathlib import Path
data_path = os.path.join(os.getcwd(), "google_transit")
print(data_path)
entries = os.listdir(data_path)
entries =  [x for x in entries if x not in ["agency.txt", "feed_info.txt"]]
dfs = {}
for entry in entries:
    dfs[entry] = (pd.read_csv(os.path.join(data_path ,entry)))

c:\Users\mikka\Desktop\ML\SI_Course\Lista1\google_transit


Zdefiniujmy klasy którymi będziemy przedstawiać nasz graf: 
1. **Edge** - klasa krawędzi
2. **Node** - klasa wierzchołka
3. **Transit Graph** - klasa grafu, agreguje klasy **Node** oraz **Edge**

In [ ]:
from dataclasses import dataclass
import datetime

@dataclass 
class Edge:
    target_stop_id: str
    departure_time: int  # sekundy od północy
    arrival_time: int
    route_name: str
    service_id: str
    is_transfer: bool = False 
@dataclass 
class TransferEdge:
    target_stop_id: str
    trasfer_time: int = 180
    is_transfer: bool = True 

@dataclass
class Node: 
    stop_id: str
    stop_name: str
    stop_lat: float
    stop_lon: float 
    type: int
    parent_station: str 

class TransitCalendar:
    def __init__(self):
        # service_id -> { 'start': 'YYYYMMDD', 'end': 'YYYYMMDD', 'days': [mon, tue, wed, thu, fri, sat, sun] }
        self.regular_schedules = {}
        # service_id -> { 'added': set(), 'removed': set() }
        self.exceptions = {}
    def set_schedules(self, schedules: dict): 
        self.regular_schedules = schedules
    def set_exceptions(self, exceptions: dict): 
        self.exceptions = exceptions

    def is_active(self, service_id: str, date_str: str) -> bool:
        """
        Sprawdza, czy pociąg faktycznie jedzie w podanym dniu.
        Format daty to YYYYMMDD, np. '20260308'.
        """
        if service_id in self.exceptions:
            if date_str in self.exceptions[service_id]['removed']:
                return False
            if date_str in self.exceptions[service_id]['added']:
                return True
                
        if service_id not in self.regular_schedules:
            return False
            
        schedule = self.regular_schedules[service_id]
        
        if not (schedule['start'] <= date_str <= schedule['end']):
            return False
            
        year = int(date_str[:4])
        month = int(date_str[4:6])
        day = int(date_str[6:8])
        weekday = datetime.date(year, month, day).weekday()        
        return schedule['days'][weekday] == 1

class TransitGraph():
    def __init__(self):
        self.nodes : dict[str ,Node] = {}
        self.adjecent : dict[str, list[Edge]] = {} # klucz id wierzchołka, wartość lista krawędzie skierowanych od tego wierzchołka
        self.calendar: TransitCalendar = {}
    def add_nodes(self, nodes: list[Node]): 
        for entry in nodes: 
            self.nodes[entry.stop_id]=entry
    def add_edge(self, source_node_id: str,edge: Edge):
        curr = self.adjecent.get(source_node_id, [])
        curr.append(edge)
        self.adjecent[source_node_id] = curr
    def set_calendar(self, calendar: TransitCalendar): 
        self.calendar= calendar
    def get_valid_neighbours(self, source_node_id: str, current_time: int, current_date: str) -> list:
        valid = [] 
        neighbours = self.adjacency_list.get(source_node_id, []) 
        ###dopisać to co chciałem doliczanie kosztu przesiadki do całości 
        for edge in neighbours:
            if getattr(edge, 'is_transfer', False):
                valid.append(edge)
                continue
                
            if edge.departure_time < current_time:
                continue
                
            if not self.calendar.is_active(edge.service_id, current_date):
                continue
                
            valid.append(edge)
            
        return valid

Przetwarzamy utworzone wcześniej Dataframy, do postaci zdefiniowanych klas


In [58]:
df_stops = dfs["stops.txt"]
df_routes = dfs["routes.txt"]
df_trips = dfs["trips.txt"]
df_stop_times = dfs["stop_times.txt"]
df_calendar  = dfs["calendar.txt"]
df_calendar_dates = dfs["calendar_dates.txt"]


def time_to_seconds(time_str: str) -> int:
    if pd.isna(time_str):
        return 0
    h, m, s = map(int, str(time_str).split(':'))
    return h * 3600 + m * 60 + s

def load_stops(df: pd.DataFrame):
    res = []
    for row in df.itertuples():
        res.append(Node(
            stop_id = row.stop_id, 
            stop_name = row.stop_name, 
            stop_lat = float(row.stop_lat), 
            stop_lon = float(row.stop_lat), 
            type = row.location_type, 
            parent_station= row.parent_station,  
        ))
    print(f"Created {len(res)} nodes")
    return res 

def load_edges(df_routes: pd.DataFrame, df_trips: pd.DataFrame, df_stop_times: pd.DataFrame, graph: 'TransitGraph'):
    routes = {}
    trips = {}
    count = 1
    
    for row in df_routes.itertuples():
        if pd.isna(row.route_short_name) or str(row.route_short_name).strip() == "":
            routes[row.route_id] = str(row.route_long_name)
        else:
            routes[row.route_id] = str(row.route_short_name)
            
    for row in df_trips.itertuples(): 
        trips[row.trip_id] = {
            "route_id": row.route_id, 
            "service_id": row.service_id
        }
        
    df_stop_times = df_stop_times.sort_values(["trip_id", "stop_sequence"])
    
    prev = None 
    for row in df_stop_times.itertuples():
        if prev and prev.trip_id == row.trip_id:
            trip_info = trips[row.trip_id]
            route_name = routes[trip_info["route_id"]]
            
            edge = Edge(
                target_stop_id=str(row.stop_id),
                departure_time=time_to_seconds(prev.departure_time), 
                arrival_time=time_to_seconds(row.arrival_time),     
                route_name=route_name,
                service_id=str(trip_info["service_id"])
            )
            
            graph.add_edge(str(prev.stop_id), edge)
            count+=1

        prev = row
    print(f"Created {count} edges")


def load_calendar(df_calendar: pd.DataFrame):
    regular_schedules = {}
    for row in df_calendar.itertuples():
        regular_schedules[str(row.service_id)] = {
            'start': str(row.start_date),
            'end': str(row.end_date),
            'days': [
                row.monday, row.tuesday, row.wednesday, 
                row.thursday, row.friday, row.saturday, row.sunday
            ]
        }
    print(f"Created {len(regular_schedules)} schedules")
    return regular_schedules

def load_calendar_dates(df_calendar_dates: pd.DataFrame):
    exceptions = {}
    for row in df_calendar_dates.itertuples():
        sid = str(row.service_id)
        date_str = str(row.date)
        ex_type = int(row.exception_type)
        
        if sid not in exceptions:
            exceptions[sid] = {'added': set(), 'removed': set()}
            
        if ex_type == 1:
            exceptions[sid]['added'].add(date_str)
        elif ex_type == 2:
            exceptions[sid]['removed'].add(date_str)
    print(f"Created {len(exceptions)} exceptions")
    return exceptions 

graph = TransitGraph()
nodes = load_stops(df_stops)
graph.add_nodes(nodes)
load_edges(df_routes, df_trips, df_stop_times, graph)
calendar = TransitCalendar()
calendar.set_exceptions(load_calendar_dates(df_calendar_dates))
calendar.set_schedules(load_calendar(df_calendar))
graph.set_calendar(calendar)
    

Created 1108 nodes
Created 45486 edges
Created 1123 exceptions
Created 3263 schedules
